# Moduł 7: CRUD Operations z SQLAlchemy

**Ćwiczenia:** #7 (CRUD Read), #8 (CRUD Write)

---

## Spis treści

1. [Wprowadzenie](#1-wprowadzenie)
2. [CREATE - Tworzenie rekordów](#2-create---tworzenie-rekordów)
3. [READ - Odczytywanie danych](#3-read---odczytywanie-danych)
4. [UPDATE - Aktualizacja danych](#4-update---aktualizacja-danych)
5. [DELETE - Usuwanie danych](#5-delete---usuwanie-danych)
6. [Filtering - Filtrowanie](#6-filtering---filtrowanie)
7. [Pagination - Stronicowanie](#7-pagination---stronicowanie)
8. [Error Handling](#8-error-handling)
9. [Best Practices](#9-best-practices)
10. [Demo Live Coding - Full CRUD API](#10-demo-live-coding---full-crud-api)
11. [Przygotowanie do ćwiczeń](#11-przygotowanie-do-ćwiczeń)
12. [Podsumowanie](#12-podsumowanie)

---

## 1. Wprowadzenie

### Czym jest CRUD?

**CRUD** = **C**reate, **R**ead, **U**pdate, **D**elete - podstawowe operacje na danych.

| Operacja | HTTP Method | SQL      | Opis                    |
|----------|-------------|----------|-------------------------|
| CREATE   | POST        | INSERT   | Tworzenie nowego rekordu |
| READ     | GET         | SELECT   | Odczytywanie danych      |
| UPDATE   | PUT/PATCH   | UPDATE   | Modyfikacja rekordu      |
| DELETE   | DELETE      | DELETE   | Usunięcie rekordu        |

### Model Task (przypomnienie)

W tym module pracujemy z prostym modelem Task z materiału 06:

```python
class Task(Base):
    __tablename__ = "tasks"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
```

### SQLAlchemy 2.0 Query API

Będziemy używać **nowoczesnego stylu** SQLAlchemy 2.0:
- `select()` zamiast `query()`
- `db.execute()` + `.scalars()`
- Type safety i lepsze IDE hints

**Przykład:**
```python
# SQLAlchemy 2.0 (✅ Preferred)
from sqlalchemy import select
stmt = select(Task).where(Task.id == 1)
result = db.execute(stmt).scalar_one()

# SQLAlchemy 1.x (⚠️ Legacy)
result = db.query(Task).filter(Task.id == 1).first()
```

---

## 2. CREATE - Tworzenie rekordów

### Podstawowa operacja INSERT

**Kroki:**
1. Stwórz instancję modelu
2. `db.add()` - dodaj do session
3. `db.commit()` - zapisz do bazy
4. `db.refresh()` - odśwież aby dostać wartości z bazy (np. id)

**Przykład:**

In [ ]:
from sqlalchemy.orm import Session
from models import Task

def create_task(db: Session, name: str) -> Task:
    """
    Stwórz nowy task

    Args:
        db: Database session
        name: Nazwa taska

    Returns:
        Utworzony task z id z bazy
    """
    # 1. Stwórz instancję (id=None)
    task = Task(name=name)
    print(f"Before add: {task.id}")  # → None

    # 2. Dodaj do session (staged for commit)
    db.add(task)
    print(f"After add: {task.id}")   # → None (jeszcze nie w bazie)

    # 3. Commit (zapisz do bazy)
    db.commit()
    print(f"After commit: {task.id}") # → None (trzeba refresh!)

    # 4. Refresh (pobierz wartości z bazy)
    db.refresh(task)
    print(f"After refresh: {task.id}") # → 1 (id z bazy!)

    return task

**Endpoint FastAPI:**

In [ ]:
from fastapi import FastAPI, Depends, status
from pydantic import BaseModel, Field

app = FastAPI()

# Request schema
class TaskCreate(BaseModel):
    name: str = Field(min_length=1, max_length=100)

# Response schema
class TaskResponse(BaseModel):
    id: int
    name: str

    class Config:
        from_attributes = True  # Dla ORM models

@app.post("/tasks", response_model=TaskResponse, status_code=status.HTTP_201_CREATED)
def create_task_endpoint(task_data: TaskCreate, db: Session = Depends(get_db)):
    """
    Stwórz nowy task

    Returns:
        201 Created z task object
    """
    task = Task(name=task_data.name)
    db.add(task)
    db.commit()
    db.refresh(task)
    return task

**Test:**
```bash
curl -X POST http://localhost:8000/tasks \
  -H "Content-Type: application/json" \
  -d '{"name": "Kupić mleko"}'

# → 201 Created
# {"id": 1, "name": "Kupić mleko"}
```

### Bulk Insert - wiele rekordów

**db.add_all()** dla wielu obiektów:

In [ ]:
def create_multiple_tasks(db: Session, names: list[str]) -> list[Task]:
    """
    Stwórz wiele tasków jednocześnie

    Args:
        db: Database session
        names: Lista nazw tasków

    Returns:
        Lista utworzonych tasków
    """
    tasks = [Task(name=name) for name in names]

    # Dodaj wszystkie na raz
    db.add_all(tasks)
    db.commit()

    # Refresh każdego (aby dostać id)
    for task in tasks:
        db.refresh(task)

    return tasks

# Użycie
# tasks = create_multiple_tasks(db, ["Task 1", "Task 2", "Task 3"])
# → [<Task(id=1, ...)>, <Task(id=2, ...)>, <Task(id=3, ...)>]

---

## 3. READ - Odczytywanie danych

### SELECT all - wszystkie rekordy

In [ ]:
from sqlalchemy import select

def get_all_tasks(db: Session) -> list[Task]:
    """
    Pobierz wszystkie taski

    Returns:
        Lista wszystkich tasków
    """
    # Sposób 1: select() (SQLAlchemy 2.0)
    stmt = select(Task)
    result = db.execute(stmt)
    tasks = result.scalars().all()

    # Sposób 2: query() (legacy, ale nadal działa)
    # tasks = db.query(Task).all()

    return tasks

**Endpoint:**

In [ ]:
@app.get("/tasks", response_model=list[TaskResponse])
def get_tasks(db: Session = Depends(get_db)):
    """
    Pobierz wszystkie taski

    Returns:
        Lista tasków
    """
    stmt = select(Task)
    tasks = db.execute(stmt).scalars().all()
    return tasks

**Test:**
```bash
curl http://localhost:8000/tasks
# → [
#     {"id": 1, "name": "Kupić mleko"},
#     {"id": 2, "name": "Zrobić zakupy"}
#   ]
```

### SELECT by ID - pojedynczy rekord

**Metody:**
- `db.get(Model, id)` - najprostsze (by primary key)
- `scalar_one()` - zwraca dokładnie 1 rekord (error jeśli 0 lub >1)
- `scalar_one_or_none()` - zwraca 1 lub None

In [ ]:
from sqlalchemy.exc import NoResultFound

def get_task_by_id(db: Session, task_id: int) -> Task | None:
    """
    Pobierz task po id

    Returns:
        Task lub None jeśli nie istnieje
    """
    # Sposób 1: db.get() (najprostszy dla primary key)
    task = db.get(Task, task_id)
    return task  # None jeśli nie istnieje

    # Sposób 2: select() + where()
    # stmt = select(Task).where(Task.id == task_id)
    # task = db.execute(stmt).scalar_one_or_none()
    # return task

**Endpoint:**

In [ ]:
from fastapi import HTTPException

@app.get("/tasks/{task_id}", response_model=TaskResponse)
def get_task(task_id: int, db: Session = Depends(get_db)):
    """
    Pobierz task po id

    Returns:
        Task object

    Raises:
        404 jeśli nie istnieje
    """
    task = db.get(Task, task_id)
    if not task:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Task {task_id} not found"
        )
    return task

**Test:**
```bash
curl http://localhost:8000/tasks/1
# → {"id": 1, "name": "Kupić mleko"}

curl http://localhost:8000/tasks/999
# → 404 Not Found: "Task 999 not found"
```

### Result methods

**Po `db.execute(stmt)`:**

| Method               | Zwraca                  | Error jeśli        |
|----------------------|-------------------------|--------------------|
| `.scalars().all()`   | `list[Model]`          | Nigdy ([] jeśli 0) |
| `.scalar_one()`      | `Model`                | 0 lub >1 results   |
| `.scalar_one_or_none()` | `Model \| None`     | >1 results         |
| `.scalars().first()` | `Model \| None`        | Nigdy              |

**Przykłady:**

In [ ]:
# .all() - lista (może być pusta)
stmt = select(Task)
tasks = db.execute(stmt).scalars().all()
# → [] jeśli brak, [task1, task2, ...] jeśli są

# .scalar_one() - dokładnie 1 (error jeśli 0 lub >1)
stmt = select(Task).where(Task.id == 1)
task = db.execute(stmt).scalar_one()
# → Task jeśli 1, NoResultFound jeśli 0, MultipleResultsFound jeśli >1

# .scalar_one_or_none() - 0 lub 1 (error jeśli >1)
stmt = select(Task).where(Task.id == 1)
task = db.execute(stmt).scalar_one_or_none()
# → Task jeśli 1, None jeśli 0, MultipleResultsFound jeśli >1

# .first() - pierwszy lub None (nigdy error)
stmt = select(Task).where(Task.name == "Test")
task = db.execute(stmt).scalars().first()
# → Task jeśli >=1, None jeśli 0

---

## 4. UPDATE - Aktualizacja danych

### ORM-style update (modyfikacja obiektu)

**Kroki:**
1. Pobierz obiekt z bazy
2. Zmodyfikuj atrybuty
3. `db.commit()` - SQLAlchemy automatycznie wykryje zmiany
4. `db.refresh()` (opcjonalne)

**Przykład:**

In [ ]:
def update_task(db: Session, task_id: int, new_name: str) -> Task | None:
    """
    Zaktualizuj task

    Args:
        db: Database session
        task_id: ID taska do aktualizacji
        new_name: Nowa nazwa

    Returns:
        Zaktualizowany task lub None jeśli nie istnieje
    """
    # 1. Pobierz obiekt
    task = db.get(Task, task_id)
    if not task:
        return None

    # 2. Zmodyfikuj
    task.name = new_name

    # 3. Commit (SQLAlchemy automatycznie wykrywa zmiany)
    db.commit()

    # 4. Refresh (opcjonalne, ale zalecane)
    db.refresh(task)

    return task

**Endpoint:**

In [ ]:
class TaskUpdate(BaseModel):
    name: str = Field(min_length=1, max_length=100)

@app.put("/tasks/{task_id}", response_model=TaskResponse)
def update_task_endpoint(task_id: int, task_data: TaskUpdate, db: Session = Depends(get_db)):
    """
    Zaktualizuj task

    Returns:
        Zaktualizowany task

    Raises:
        404 jeśli nie istnieje
    """
    # Pobierz
    task = db.get(Task, task_id)
    if not task:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Task {task_id} not found"
        )

    # Zmodyfikuj
    task.name = task_data.name

    # Commit
    db.commit()
    db.refresh(task)

    return task

**Test:**
```bash
curl -X PUT http://localhost:8000/tasks/1 \
  -H "Content-Type: application/json" \
  -d '{"name": "Kupić chleb"}'

# → {"id": 1, "name": "Kupić chleb"}
```

### Partial update (PATCH)

**PUT vs PATCH:**
- **PUT** - pełna zamiana (wszystkie pola)
- **PATCH** - częściowa aktualizacja (tylko wybrane pola)

**Przykład PATCH:**

In [ ]:
class TaskPatch(BaseModel):
    """Schema dla częściowej aktualizacji"""
    name: str | None = None

@app.patch("/tasks/{task_id}", response_model=TaskResponse)
def patch_task(task_id: int, task_data: TaskPatch, db: Session = Depends(get_db)):
    """
    Częściowa aktualizacja taska (PATCH)

    Tylko przekazane pola zostaną zaktualizowane
    """
    task = db.get(Task, task_id)
    if not task:
        raise HTTPException(status_code=404, detail="Task not found")

    # Aktualizuj tylko podane pola
    update_data = task_data.model_dump(exclude_unset=True)
    for key, value in update_data.items():
        setattr(task, key, value)

    db.commit()
    db.refresh(task)
    return task

---

## 5. DELETE - Usuwanie danych

### db.delete() - usunięcie obiektu

In [ ]:
def delete_task(db: Session, task_id: int) -> bool:
    """
    Usuń task

    Args:
        db: Database session
        task_id: ID taska do usunięcia

    Returns:
        True jeśli usunięto, False jeśli nie istniał
    """
    # 1. Pobierz obiekt
    task = db.get(Task, task_id)
    if not task:
        return False

    # 2. Usuń
    db.delete(task)

    # 3. Commit
    db.commit()

    return True

**Endpoint:**

In [ ]:
@app.delete("/tasks/{task_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_task_endpoint(task_id: int, db: Session = Depends(get_db)):
    """
    Usuń task

    Returns:
        204 No Content jeśli usunięto

    Raises:
        404 jeśli nie istnieje
    """
    task = db.get(Task, task_id)
    if not task:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Task {task_id} not found"
        )

    db.delete(task)
    db.commit()

    # 204 No Content - brak body!
    return

**Test:**
```bash
curl -X DELETE http://localhost:8000/tasks/1
# → 204 No Content (brak body)

curl -X DELETE http://localhost:8000/tasks/999
# → 404 Not Found
```

---

## 6. Filtering - Filtrowanie

### .where() - warunki SQL

**Podstawowe operatory:**

In [ ]:
from sqlalchemy import select

# Równość
stmt = select(Task).where(Task.id == 1)

# Nierówność
stmt = select(Task).where(Task.id != 1)

# Większe/mniejsze
stmt = select(Task).where(Task.id > 5)
stmt = select(Task).where(Task.id >= 5)

# IN (lista wartości)
stmt = select(Task).where(Task.id.in_([1, 2, 3]))

# LIKE (pattern matching)
stmt = select(Task).where(Task.name.like("%mleko%"))

# ILIKE (case-insensitive LIKE)
stmt = select(Task).where(Task.name.ilike("%MLEKO%"))

# IS NULL / IS NOT NULL
stmt = select(Task).where(Task.name.is_(None))
stmt = select(Task).where(Task.name.is_not(None))

### Multiple conditions - AND, OR

**AND (wszystkie warunki):**

In [ ]:
from sqlalchemy import and_, or_

# Sposób 1: Wiele .where() (AND)
stmt = select(Task).where(Task.id > 5).where(Task.name.like("%test%"))
# → WHERE id > 5 AND name LIKE '%test%'

# Sposób 2: and_()
stmt = select(Task).where(and_(Task.id > 5, Task.name.like("%test%")))
# → WHERE id > 5 AND name LIKE '%test%'

**OR (dowolny warunek):**

In [ ]:
# OR
stmt = select(Task).where(or_(Task.id == 1, Task.id == 2))
# → WHERE id = 1 OR id = 2

# Kombinacja AND + OR
stmt = select(Task).where(
    and_(
        Task.id > 5,
        or_(Task.name.like("%test%"), Task.name.like("%demo%"))
    )
)
# → WHERE id > 5 AND (name LIKE '%test%' OR name LIKE '%demo%')

### Search endpoint - przykład

In [ ]:
@app.get("/tasks/search", response_model=list[TaskResponse])
def search_tasks(q: str | None = None, db: Session = Depends(get_db)):
    """
    Wyszukaj taski po nazwie

    Args:
        q: Search query (case-insensitive)

    Returns:
        Lista pasujących tasków
    """
    stmt = select(Task)

    # Jeśli query podane, filtruj
    if q:
        stmt = stmt.where(Task.name.ilike(f"%{q}%"))

    tasks = db.execute(stmt).scalars().all()
    return tasks

**Test:**
```bash
curl "http://localhost:8000/tasks/search?q=mleko"
# → [{"id": 1, "name": "Kupić mleko"}]

curl "http://localhost:8000/tasks/search"
# → Wszystkie taski
```

---

## 7. Pagination - Stronicowanie

### .limit() i .offset()

In [ ]:
def get_tasks_paginated(db: Session, skip: int = 0, limit: int = 10) -> list[Task]:
    """
    Pobierz taski z paginacją

    Args:
        db: Database session
        skip: Ile rekordów pominąć (offset)
        limit: Max liczba rekordów

    Returns:
        Lista tasków dla danej strony
    """
    stmt = select(Task).offset(skip).limit(limit)
    tasks = db.execute(stmt).scalars().all()
    return tasks

### Pagination dependency (z modułu 05)

In [ ]:
from fastapi import Query

def pagination(
    page: int = Query(1, ge=1, description="Numer strony"),
    page_size: int = Query(10, ge=1, le=100, description="Rozmiar strony")
):
    """Dependency: Pagination parameters"""
    skip = (page - 1) * page_size
    return {"skip": skip, "limit": page_size, "page": page}

@app.get("/tasks", response_model=list[TaskResponse])
def get_tasks_endpoint(p: dict = Depends(pagination), db: Session = Depends(get_db)):
    """
    Pobierz taski z paginacją

    Query params:
        page: Numer strony (od 1)
        page_size: Rozmiar strony (1-100)
    """
    stmt = select(Task).offset(p["skip"]).limit(p["limit"])
    tasks = db.execute(stmt).scalars().all()
    return tasks

**Test:**
```bash
curl "http://localhost:8000/tasks?page=1&page_size=10"
# → Strona 1 (taski 1-10)

curl "http://localhost:8000/tasks?page=2&page_size=10"
# → Strona 2 (taski 11-20)
```

### Sortowanie - .order_by()

In [ ]:
from sqlalchemy import desc, asc

# Sortowanie rosnąco (domyślne)
stmt = select(Task).order_by(Task.name)
# → ORDER BY name ASC

# Sortowanie malejąco
stmt = select(Task).order_by(desc(Task.name))
# → ORDER BY name DESC

# Wiele kolumn
stmt = select(Task).order_by(Task.name, desc(Task.id))
# → ORDER BY name ASC, id DESC

### Count - liczba rekordów

In [ ]:
from sqlalchemy import func, select

def count_tasks(db: Session) -> int:
    """Policz wszystkie taski"""
    stmt = select(func.count()).select_from(Task)
    count = db.execute(stmt).scalar()
    return count

# Alternatywnie (SQLAlchemy 1.x style)
# count = db.query(Task).count()

---

## 8. Error Handling

### IntegrityError - naruszenie constraints

**Problem:** Duplicate unique key, foreign key violation, etc.

In [ ]:
from sqlalchemy.exc import IntegrityError

@app.post("/tasks", response_model=TaskResponse, status_code=201)
def create_task_safe(task_data: TaskCreate, db: Session = Depends(get_db)):
    """
    Stwórz task z obsługą błędów
    """
    try:
        task = Task(name=task_data.name)
        db.add(task)
        db.commit()
        db.refresh(task)
        return task
    except IntegrityError as e:
        db.rollback()  # Cofnij transakcję
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=f"Database integrity error: {str(e.orig)}"
        )

### Rollback - cofnięcie transakcji

**Kiedy używać:**
- Po błędzie (IntegrityError, etc.)
- Gdy walidacja biznesowa nie przeszła
- Gdy chcesz anulować zmiany

**Przykład:**

In [ ]:
def create_task_with_validation(db: Session, name: str) -> Task:
    """
    Stwórz task z własną walidacją biznesową
    """
    # Sprawdź czy task o tej nazwie już istnieje
    stmt = select(Task).where(Task.name == name)
    existing = db.execute(stmt).scalar_one_or_none()

    if existing:
        raise HTTPException(
            status_code=400,
            detail=f"Task '{name}' already exists"
        )

    # Stwórz nowy task
    task = Task(name=name)
    db.add(task)
    db.commit()
    db.refresh(task)
    return task

---

## 9. Best Practices

### ✅ Rób tak:

**1. Zawsze używaj db.refresh() po commit:**
```python
# ✅ Dobre - refresh pobiera wartości z bazy
task = Task(name="Test")
db.add(task)
db.commit()
db.refresh(task)  # task.id teraz ma wartość!
return task

# ❌ Złe - brak id
task = Task(name="Test")
db.add(task)
db.commit()
return task  # task.id = None!
```

**2. Używaj select() (SQLAlchemy 2.0):**
```python
# ✅ Dobre - nowoczesny styl
stmt = select(Task).where(Task.id == 1)
task = db.execute(stmt).scalar_one()

# ⚠️ Legacy - działa, ale przestarzałe
task = db.query(Task).filter(Task.id == 1).first()
```

**3. Obsługuj 404:**
```python
# ✅ Dobre - jasny error
task = db.get(Task, task_id)
if not task:
    raise HTTPException(404, f"Task {task_id} not found")

# ❌ Złe - zwraca None bez info
task = db.get(Task, task_id)
return task  # None jeśli nie istnieje
```

**4. DELETE zwraca 204 No Content:**
```python
# ✅ Dobre
@app.delete("/tasks/{id}", status_code=204)
def delete_task(...):
    # ...
    return  # Brak body

# ❌ Złe - 204 nie powinno mieć body
@app.delete("/tasks/{id}", status_code=204)
def delete_task(...):
    return {"message": "Deleted"}  # Błąd!
```

**5. Używaj type hints:**
```python
# ✅ Dobre
def get_task(db: Session, task_id: int) -> Task | None:
    return db.get(Task, task_id)

# ❌ Złe - brak type hints
def get_task(db, task_id):
    return db.get(Task, task_id)
```

### ❌ Nie rób tak:

**1. Nie zapominaj commit:**
```python
# ❌ Złe - brak commit, zmiany nie zapisane!
task = Task(name="Test")
db.add(task)
# Brak db.commit()!
return task  # Nie zapisane w bazie!
```

**2. Nie używaj raw SQL bez potrzeby:**
```python
# ❌ Złe - raw SQL
db.execute("DELETE FROM tasks WHERE id = :id", {"id": task_id})

# ✅ Dobre - ORM
task = db.get(Task, task_id)
db.delete(task)
db.commit()
```

**3. Nie mieszaj .all() z .scalar_one():**
```python
# ❌ Złe - .all() dla pojedynczego rekordu
stmt = select(Task).where(Task.id == 1)
tasks = db.execute(stmt).scalars().all()  # Zwraca listę!
task = tasks[0]  # Może być IndexError

# ✅ Dobre - odpowiednia metoda
task = db.get(Task, 1)  # Pojedynczy rekord
```

---

## 12. Podsumowanie

### Kluczowe zagadnienia:

1. **CREATE** - `db.add()` + `db.commit()` + `db.refresh()`
2. **READ** - `select()` + `db.execute()` + `.scalars()`
3. **UPDATE** - Modyfikacja obiektu + `db.commit()`
4. **DELETE** - `db.delete()` + `db.commit()`
5. **Filtering** - `.where()` z warunkami (==, !=, >, <, in_, like, ilike)
6. **Pagination** - `.offset()` + `.limit()`
7. **Sorting** - `.order_by()` z asc/desc
8. **Error Handling** - HTTPException dla 404, IntegrityError dla constraint violations

### SQLAlchemy 2.0 patterns:

```python
# SELECT all
stmt = select(Task)
tasks = db.execute(stmt).scalars().all()

# SELECT by ID
task = db.get(Task, task_id)

# SELECT with conditions
stmt = select(Task).where(Task.name.like("%test%"))
tasks = db.execute(stmt).scalars().all()

# INSERT
task = Task(name="Test")
db.add(task)
db.commit()
db.refresh(task)

# UPDATE
task.name = "Updated"
db.commit()

# DELETE
db.delete(task)
db.commit()
```

### Następny krok:
Przejdź do **Moduł 08: Relationships** - relacje między tabelami (User → Tasks)